In [4]:
# In een notebook cel:
%pip install pettingzoo[atari] multi-agent-ale-py autorom[accept-rom-license]
%pip install tensorflow tensorflow-probability numpy matplotlib scikit-learn pandas

# Download Atari ROMs (alleen de eerste keer nodig)
!AutoROM -y

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 14.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.8/707.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 60.0 MB/s eta 0:00:00
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446710 sha256=2d5bf1a3d279a0483f4d387f92c9e9c6399f77050f2dea71cea51188577c40e3
  Stored in directory: /root/.cache/pip/wheels/99/f1/ff/c6966c034a8259164bdc9deb4d1ea839f119474638100e6645
Successfully built AutoROM.accept-rom-license
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms
	/usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms

Existing 

In [8]:
# Imports
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp

def build_actor_network(state_shape, action_dim, hidden_layers=[256, 128]):
    inputs = tf.keras.Input(shape=state_shape)
    x = inputs

    for units in hidden_layers:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(action_dim, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def build_critic_network(state_shape, hidden_layers=[256, 128]):
    inputs = tf.keras.Input(shape=state_shape)
    x = inputs

    for units in hidden_layers:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(1, activation="linear")(x)
    return tf.keras.Model(inputs, outputs)

# A2C Agent Implementation
class A2CAgent:
    def __init__(self, state_dim, action_dim, player_id,
             learning_rate=0.001, gamma=0.99, entropy_coef=0.01,
             value_coef=0.5, max_grad_norm=0.5, hidden_layers=[256, 128]):
        self.player_id = player_id
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef
        self.max_grad_norm = max_grad_norm

        self.actor = build_actor_network(state_dim, action_dim, hidden_layers)
        self.critic = build_critic_network(state_dim, hidden_layers)
        self.actor_optimizer = tf.keras.optimizers.Adam(learning_rate)
        self.critic_optimizer = tf.keras.optimizers.Adam(learning_rate)

        # Training metrics
        self.episode_rewards = []
        self.episode_losses = []

    def select_action(self, state):
        # Policy-based actie selectie
        state = np.expand_dims(state, axis=0) # Belangrijk voor Atari
        probs = self.actor(state)
        dist = tfp.distributions.Categorical(probs=probs[0])
        action = dist.sample()
        # Ensure action is within valid range [0, action_dim - 1]
        action = tf.clip_by_value(action, 0, self.actor.output_shape[-1] - 1)
        log_prob = dist.log_prob(action)
        return int(action.numpy()), log_prob

    def update(self, states, actions, rewards, dones, next_states):
        states = np.array(states)
        actions = np.array(actions)

        # Compute returns
        returns = []
        R = 0
        for r, d in zip(reversed(rewards), reversed(dones)):
            if d:
                R = 0
            R = r + self.gamma * R
            returns.insert(0, R)
        returns = tf.convert_to_tensor(returns, dtype=tf.float32)

        # Compute values and advantages
        values = tf.squeeze(self.critic(states))
        values = tf.cast(values, tf.float32)
        advantages = returns - values

        # Update actor
        with tf.GradientTape() as tape:
            probs = self.actor(states)
            dist = tfp.distributions.Categorical(probs=probs)
            actions_tensor = tf.convert_to_tensor(actions, dtype=tf.int32)
            log_probs = dist.log_prob(actions_tensor)

            actor_loss = -tf.reduce_mean(log_probs * tf.stop_gradient(advantages))
            entropy = tf.reduce_mean(dist.entropy())
            total_actor_loss = actor_loss - self.entropy_coef * entropy

        actor_grads = tape.gradient(total_actor_loss, self.actor.trainable_variables)
        actor_grads, _ = tf.clip_by_global_norm(actor_grads, self.max_grad_norm)
        self.actor_optimizer.apply_gradients(zip(actor_grads, self.actor.trainable_variables))

        # Update critic
        with tf.GradientTape() as tape:
            values_pred = tf.squeeze(self.critic(states))
            critic_loss = tf.reduce_mean(tf.square(returns - values_pred))

        critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
        critic_grads, _ = tf.clip_by_global_norm(critic_grads, self.max_grad_norm)
        self.critic_optimizer.apply_gradients(zip(critic_grads, self.critic.trainable_variables))

        return total_actor_loss.numpy(), critic_loss.numpy()

    def train_episode(self, env, agent_name_unused):
        states, actions_list, rewards_list, dones_list = [], [], [], []
        total_reward = 0

        # Reset environment and get initial observations for all agents
        all_observations, _ = env.reset()

        # Get the specific agent ID this A2CAgent instance is responsible for
        current_agent_id = env.agents[self.player_id]
        current_obs = all_observations[current_agent_id]

        # Loop until the current_agent_id's episode is done
        while current_agent_id in env.agents:
            # Agent selects its action
            action_for_self, _ = self.select_action(current_obs)

            # Collect actions for all active agents.
            # The agent being trained uses its policy.
            # Other active agents take random actions.
            actions_dict = {}
            for agent_id in env.agents:
                if agent_id == current_agent_id:
                    actions_dict[agent_id] = action_for_self
                else:
                    # For other agents, take a random action
                    actions_dict[agent_id] = env.action_space(agent_id).sample()

            # Step the environment with all collected actions
            next_all_observations, all_rewards, all_terminations, all_truncations, _ = env.step(actions_dict)

            # Extract the experience relevant to the current agent
            reward_for_self = all_rewards[current_agent_id]
            terminated_for_self = all_terminations[current_agent_id]
            truncated_for_self = all_truncations[current_agent_id]

            # Store the experience
            states.append(current_obs)
            actions_list.append(action_for_self)
            rewards_list.append(reward_for_self)
            dones_list.append(terminated_for_self or truncated_for_self)

            total_reward += reward_for_self

            # Update current_obs for the next step, but only if the agent is still alive
            if current_agent_id in next_all_observations:
                current_obs = next_all_observations[current_agent_id]
            else:
                # If current_agent_id is no longer in next_all_observations, it means it's done.
                break

        # Update networks
        if len(states) > 0:
            actor_loss, critic_loss = self.update(states, actions_list, rewards_list, dones_list, states)
            self.episode_rewards.append(total_reward)
            self.episode_losses.append((actor_loss, critic_loss))

        return total_reward

    def evaluate(self, env, n_episodes=10):
        eval_rewards = []

        for _ in range(n_episodes):
            all_observations, _ = env.reset()
            current_agent_id = env.agents[self.player_id]
            current_obs = all_observations[current_agent_id]

            episode_reward = 0

            # Loop until the current_agent_id's episode is done
            while current_agent_id in env.agents:
                action_for_self, _ = self.select_action(current_obs)

                actions_dict = {}
                for agent_id in env.agents:
                    if agent_id == current_agent_id:
                        actions_dict[agent_id] = action_for_self
                    else:
                        # Other agents take random actions during evaluation
                        actions_dict[agent_id] = env.action_space(agent_id).sample()

                next_all_observations, all_rewards, all_terminations, all_truncations, _ = env.step(actions_dict)

                reward_for_self = all_rewards[current_agent_id]

                episode_reward += reward_for_self

                if current_agent_id in next_all_observations:
                    current_obs = next_all_observations[current_agent_id]
                else:
                    break

            eval_rewards.append(episode_reward)

        return np.mean(eval_rewards), np.std(eval_rewards)

In [9]:
# Importeer libraries
from pettingzoo.atari import warlords_v3 # Importeer warlords_v3 direct
import numpy as np
import matplotlib.pyplot as plt

# Maak environment aan (RAM mode)
env = warlords_v3.parallel_env(obs_type="ram") # Gebruik parallel_env voor meerdere agents

# Reset environment om shapes te bepalen
# Voor parallelle omgevingen retourneert reset dictionaries van observaties en info
observations, infos = env.reset()
# Haal de observatie voor de eerste agent op (aangenomen dat de A2C agent voor één speler is)
obs = observations[env.agents[0]]

state_dim = obs.shape  # (128,) voor RAM
action_dim = env.action_space(env.agents[0]).n  # Aantal acties (meestal 18)

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Agents: {env.agents}")

State dimension: (128,)
Action dimension: 6
Agents: ['first_0', 'second_0', 'third_0', 'fourth_0']


In [10]:
# Maak A2C agent aan met hyperparameters
a2c_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=int(action_dim),  # Expliciet casten naar int
    player_id=0,  # Of de juiste player_id voor jouw agent
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    hidden_layers=[256, 128]
)

print("A2C Agent geïnitialiseerd!")

A2C Agent geïnitialiseerd!


In [ ]:
# Training parameters
num_episodes = 500
eval_interval = 50  # Evalueer elke X episodes
save_interval = 100  # Sla model op elke X episodes

# Training loop
print(f"Start training voor {num_episodes} episodes...")

for episode in range(num_episodes):
    # Train één episode
    episode_reward = a2c_agent.train_episode(env, "A2C")

    # Print voortgang
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(a2c_agent.episode_rewards[-10:])
        print(f"Episode {episode+1}/{num_episodes} | "
              f"Reward: {episode_reward:.2f} | "
              f"Avg (last 10): {avg_reward:.2f} | "
              f"Total Episodes: {len(a2c_agent.episode_rewards)}")

    # Evalueer periodiek
    if (episode + 1) % eval_interval == 0:
        mean_reward, std_reward = a2c_agent.evaluate(env, n_episodes=10)
        print(f"  --- Evaluation: {mean_reward:.2f} ± {std_reward:.2f} ---")

    # Sla model op periodiek
    if (episode + 1) % save_interval == 0:
        a2c_agent.actor.save_weights(f'a2c_actor_ep{episode+1}.h5')
        a2c_agent.critic.save_weights(f'a2c_critic_ep{episode+1}.h5')
        print(f"  --- Model saved at episode {episode+1} ---")

print("Training voltooid!")

Start training voor 500 episodes...


In [ ]:
print('Checking for GPU availability:')
print(tf.config.list_physical_devices('GPU'))

if len(tf.config.list_physical_devices('GPU')) > 0:
    print('GPU is available and being used. Great!')
else:
    print('No GPU found. Training will be significantly slower. Consider changing the runtime type to GPU (Runtime > Change runtime type > Hardware accelerator: GPU).')


In [ ]:
# Finale evaluatie
final_mean, final_std = a2c_agent.evaluate(env, n_episodes=20)
print(f"\nFinale Evaluatie:")
print(f"Mean Reward: {final_mean:.2f} ± {final_std:.2f}")
print(f"Total training episodes: {len(a2c_agent.episode_rewards)}")

In [ ]:
def plot_training_curves(agent):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Rewards
    rewards = agent.episode_rewards
    axes[0, 0].plot(rewards, alpha=0.3, label='Raw Rewards')

    # Moving average
    window = 20
    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0, 0].plot(range(window-1, len(rewards)), moving_avg,
                        label=f'Moving Avg (window={window})', linewidth=2)

    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_title('Training Rewards')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Losses
    if agent.episode_losses:
        actor_losses = [loss[0] for loss in agent.episode_losses]
        critic_losses = [loss[1] for loss in agent.episode_losses]

        axes[0, 1].plot(actor_losses, label='Actor Loss', alpha=0.7)
        axes[0, 1].plot(critic_losses, label='Critic Loss', alpha=0.7)
        axes[0, 1].set_xlabel('Episode')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Training Losses')
        axes[0, 1].legend()
        axes[0, 1].grid(True)

    # Reward distribution
    axes[1, 0].hist(rewards, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Reward')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Reward Distribution')
    axes[1, 0].grid(True)

    # Training stability (rolling std)
    if len(rewards) >= window:
        rolling_std = []
        for i in range(window, len(rewards)+1):
            rolling_std.append(np.std(rewards[i-window:i]))
        axes[1, 1].plot(range(window, len(rewards)+1), rolling_std)
        axes[1, 1].set_xlabel('Episode')
        axes[1, 1].set_ylabel('Rolling Std Dev')
        axes[1, 1].set_title(f'Training Stability (window={window})')
        axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

# Plot de resultaten
plot_training_curves(a2c_agent)

In [ ]:
# Sla het finale model op
a2c_agent.actor.save_weights('a2c_actor_final.h5')
a2c_agent.critic.save_weights('a2c_critic_final.h5')

# Sla ook de hyperparameters op
import json
hyperparams = {
    'learning_rate': 0.001,
    'gamma': 0.99,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'max_grad_norm': 0.5,
    'hidden_layers': [256, 128],
    'num_episodes': len(a2c_agent.episode_rewards),
    'final_mean_reward': final_mean,
    'final_std_reward': final_std
}

with open('a2c_hyperparams.json', 'w') as f:
    json.dump(hyperparams, f, indent=4)

print("Model en hyperparameters opgeslagen!")

In [ ]:
# Laad een getraind model
loaded_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

# Laad de weights
loaded_agent.actor.load_weights('a2c_actor_final.h5')
loaded_agent.critic.load_weights('a2c_critic_final.h5')

print("Model geladen!")

In [ ]:
# In het tournament notebook, vervang de PPO agent import:
from A2C_agent import A2CAgent

# Laad het getrainde model
a2c_trained = A2CAgent(
    state_dim=(128,),
    action_dim=18,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

a2c_trained.actor.load_weights('a2c_actor_final.h5')
a2c_trained.critic.load_weights('a2c_critic_final.h5')

# Voeg toe aan agent_instances
agent_instances = [
    RandomAgent(),
    HeuristicAgent(),
    a2c_trained,  # Je getrainde A2C agent
    RandomAgent()
]

agent_names = ['RandomAgent', 'HeuristicAgent', 'A2CAgent', 'Agent4']